<a href="https://colab.research.google.com/github/rajeshkannan290208-create/FUTURE_ML_03/blob/main/project3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# =========================================
# RESUME SCREENING & RANKING SYSTEM
# =========================================

import pandas as pd
import numpy as np
import re
import nltk
import kagglehub
import os

from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

# =========================================
# DOWNLOAD DATASET
# =========================================
path = kagglehub.dataset_download("snehaanbhawal/resume-dataset")
print("Dataset Path:", path)

print("Main Folder:", os.listdir(path))

resume_folder = os.path.join(path, "Resume")
print("Inside Resume Folder:", os.listdir(resume_folder))

file_path = os.path.join(resume_folder, "Resume.csv")

df = pd.read_csv(file_path)

print("\nDataset Loaded Successfully!")
print(df.head())

# =========================================
# SELECT COLUMN
# =========================================
df = df[['Resume_str']]
df.dropna(inplace=True)

print("Dataset Shape:", df.shape)

# =========================================
# TEXT CLEANING
# =========================================
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\W', ' ', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    return " ".join(words)

df['clean_resume'] = df['Resume_str'].apply(clean_text)

# =========================================
# JOB DESCRIPTION
# =========================================
job_description = """
Looking for a Python developer with machine learning, data analysis, and SQL skills.
"""

job_clean = clean_text(job_description)

# =========================================
# TF-IDF
# =========================================
vectorizer = TfidfVectorizer(max_features=5000)

all_text = df['clean_resume'].tolist() + [job_clean]
X = vectorizer.fit_transform(all_text)

resume_vectors = X[:-1]
job_vector = X[-1]

# =========================================
# SIMILARITY
# =========================================
scores = cosine_similarity(resume_vectors, job_vector)

# FIX: flatten scores to 1D
df['score'] = scores.flatten()

# =========================================
# SORT
# =========================================
df_sorted = df.sort_values(by='score', ascending=False)

print("\n===== TOP 5 CANDIDATES =====")
print(df_sorted[['Resume_str', 'score']].head(5))

# =========================================
# SKILL EXTRACTION
# =========================================
skills_list = [
    'python', 'machine learning', 'sql', 'data analysis',
    'java', 'c++', 'deep learning', 'excel', 'power bi'
]

def extract_skills(text):
    return [skill for skill in skills_list if skill in text.lower()]

df_sorted['skills'] = df_sorted['Resume_str'].apply(extract_skills)

# =========================================
# MISSING SKILLS
# =========================================
job_skills = extract_skills(job_description)

def missing_skills(candidate_skills):
    return list(set(job_skills) - set(candidate_skills))

df_sorted['missing_skills'] = df_sorted['skills'].apply(missing_skills)

# =========================================
# FINAL OUTPUT
# =========================================
print("\n===== FINAL RANKED RESULTS =====")

top_n = 5
for rank, (idx, row) in enumerate(df_sorted.head(top_n).iterrows(), start=1):
    print("\n---------------------------")
    print("Rank:", rank)
    print("Score:", round(row['score'], 3))
    print("Skills:", row['skills'])
    print("Missing Skills:", row['missing_skills'])

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Using Colab cache for faster access to the 'resume-dataset' dataset.
Dataset Path: /kaggle/input/resume-dataset
Main Folder: ['Resume', 'data']
Inside Resume Folder: ['Resume.csv']

Dataset Loaded Successfully!
         ID                                         Resume_str  \
0  16852973           HR ADMINISTRATOR/MARKETING ASSOCIATE\...   
1  22323967           HR SPECIALIST, US HR OPERATIONS      ...   
2  33176873           HR DIRECTOR       Summary      Over 2...   
3  27018550           HR SPECIALIST       Summary    Dedica...   
4  17812897           HR MANAGER         Skill Highlights  ...   

                                         Resume_html Category  
0  <div class="fontsize fontface vmargins hmargin...       HR  
1  <div class="fontsize fontface vmargins hmargin...       HR  
2  <div class="fontsize fontface vmargins hmargin...       HR  
3  <div class="fontsize fontface vmargins hmargin...       HR  
4  <div class="fontsize fontface vmargins hmargin...       HR  
Dataset 